# H1 · Interpolation error on degraded CIRRUS

Issue [#18](https://github.com/fedro86/DECADE/issues/18). CIRRUS (1 km / 5 min) is degraded as if it were
ODYSSEY: mean in linear Z over 2 × 2 blocks (`opera_ds.validate.degrade_frame`), one frame every 15 min.
The degraded frames are interpolated back to 1 km / 5 min with the methods of `opera_ds.interp` and
compared with the real CIRRUS. `run_validation` runs unchanged: only its reader of the coarse files is
`read_degraded`. The truth is known, so the error is the interpolation error alone, without the
ODYSSEY–CIRRUS product difference (H3).

Choices still open, asked in #18:

- **time shift:** none. The degraded frame at NT is CIRRUS at NT, so `alpha` = 0 is a real (degraded) frame;
- **event type:** provisional rule, a time is *convective* when at least `CONV_SHARE` of the echo
  (≥ 10 dBZ) over Belgium is ≥ `CONV_DBZ`.

In [ ]:
%load_ext autoreload
%autoreload 2
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import csv
import datetime as dt
import functools
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

from opera_ds.odim_io import index_files, read_odim_composite
from opera_ds.paths import REPO_DIR
from opera_ds.validate import read_degraded, run_validation

## Parameters

In [ ]:
START = dt.datetime(2025, 9, 8)        # one week for the prototype (rainy, with convection)
END = dt.datetime(2025, 9, 15)         # excluded
METHODS = ("nearest", "linear", "lk")
WORKERS = 12
CONV_DBZ, CONV_SHARE = 40.0, 0.02      # provisional event-type rule
OUT = RESULTS_DIR / "h1_degraded_cirrus" / f"{START:%Y%m%d}-{END:%Y%m%d}"
OUT.mkdir(parents=True, exist_ok=True)

## Files and Belgium mask

In [ ]:
cirrus = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
margin = dt.timedelta(hours=1)         # motion context around the period
cirrus = {t: p for t, p in cirrus.items() if START - margin <= t < END + margin}
coarse = {t: p for t, p in cirrus.items() if t.minute % 15 == 0}
print(f"CIRRUS frames: {len(cirrus)}, degraded frames (every 15 min): {len(coarse)}")

ref = read_odim_composite(next(iter(cirrus.values())))
r0, c0 = int(ref.where["crop_row0"]), int(ref.where["crop_col0"])
ny, nx = ref.values.shape
region = np.load(DATA_DIR / "mask_BEL_1km.npy")[r0:r0 + ny, c0:c0 + nx]
np.save(OUT / "region_mask_crop.npy", region)
print(f"crop {ny} x {nx} px at 1 km; Belgium {region.sum()} px")

## Validation against the real CIRRUS

In [ ]:
t = dt.datetime.now()
run_validation(coarse, cirrus, START, END, str(OUT), shifts=(0.0,), methods=METHODS,
               pair_stride=1, workers=WORKERS, region_mask_path=str(OUT / "region_mask_crop.npy"),
               reader=functools.partial(read_degraded, factor=2), cirrus_reader=read_odim_composite)
print(f"done in {(dt.datetime.now() - t).total_seconds() / 60:.1f} min; outputs in {OUT}")

In [ ]:
def read_csv(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

summary = read_csv(OUT / "summary.csv")
cols = ("rmse", "bias", "corr", "csi_20", "csi_30", "fss_20dBZ_11km", "hf_energy_ratio")
print(f"{'method':8s} {'alpha':>6s} {'times':>6s} " + " ".join(f"{c:>15s}" for c in cols))
for r in sorted(summary, key=lambda r: (r["method"], float(r["alpha"]))):
    print(f"{r['method']:8s} {float(r['alpha']):6.3f} {int(r['n_times']):6d} "
          + " ".join(f"{float(r[c]):15.3f}" for c in cols))

## By `alpha`, against the 4 days of real ODYSSEY

`alpha` = 0 is a real frame: its error is the floor (upscaling only, here). The extra error at
`alpha` = 1/3 and 2/3 is the interpolation error. Both runs are taken at shift 0, so the `alpha`
values match. Real ODYSSEY has no `alpha` = 0: on 1–5/7/2024 CIRRUS is missing at the quarter
hours (`docs/data.md`, *Overlap between products*).

In [ ]:
ref4 = [r for r in read_csv(REPO_DIR / "docs" / "figures" / "summary.csv") if float(r["shift_min"]) == 0.0]

def rmse_by_alpha(rows, m):
    return {round(float(r["alpha"]), 3): float(r["rmse"]) for r in rows if r["method"] == m}

print(f"{'method':8s} {'':22s} {'a=0':>7s} {'a=1/3':>7s} {'a=2/3':>7s} {'extra':>7s}")
for m in METHODS:
    for name, rows in (("degraded CIRRUS, 1 wk", summary), ("real ODYSSEY, 4 days", ref4)):
        d = rmse_by_alpha(rows, m)
        a0, a1, a2 = (d.get(a, np.nan) for a in (0.0, 0.333, 0.667))
        extra = np.mean([a1, a2]) - a0
        print(f"{m:8s} {name:22s} {a0:7.2f} {a1:7.2f} {a2:7.2f} {extra:7.2f}")

## By scale (interpolated times)

In [ ]:
fss_cols = [c for c in summary[0] if c.startswith("fss_20dBZ_")]
r_cols = [c for c in summary[0] if c.startswith("r") and c.endswith("_rmse")]
print(f"{'method':8s} {'alpha':>6s} " + " ".join(f"{c[6:]:>8s}" for c in fss_cols) + " | "
      + " ".join(f"{c:>9s}" for c in r_cols))
for r in sorted(summary, key=lambda r: (r["method"], float(r["alpha"]))):
    print(f"{r['method']:8s} {float(r['alpha']):6.3f} " + " ".join(f"{float(r[c]):8.3f}" for c in fss_cols)
          + " | " + " ".join(f"{float(r[c]):9.3f}" for c in r_cols))
print("FSS at 20 dBZ by window (km); RMSE of R (mm/h) by block (km)")

## By event type (provisional rule)

In [ ]:
def conv_share(path):
    fr = read_odim_composite(path)
    echo = fr.coverage & ~fr.undetect & (fr.values >= 10.0) & region
    return (echo & (fr.values >= CONV_DBZ)).sum() / max(echo.sum(), 1)

per_time = read_csv(OUT / "per_time.csv")
share = {t: conv_share(cirrus[dt.datetime.fromisoformat(t)]) for t in sorted({r["time"] for r in per_time})}
kind = {t: "convective" if s >= CONV_SHARE else "stratiform" for t, s in share.items()}
print({k: sum(v == k for v in kind.values()) for k in ("convective", "stratiform")}, "times")

agg = defaultdict(lambda: [0, 0.0])
for r in per_time:
    n = int(r["n_wet"])
    if n and r["rmse"] not in ("", "nan"):
        a = agg[(r["method"], round(float(r["alpha"]), 3), kind[r["time"]])]
        a[0] += n
        a[1] += n * float(r["rmse"]) ** 2
rmse_kind = {k: np.sqrt(v[1] / v[0]) for k, v in agg.items()}
print(f"{'method':8s} {'type':11s} {'a=0':>7s} {'a=1/3':>7s} {'a=2/3':>7s}")
for m in METHODS:
    for k in ("convective", "stratiform"):
        print(f"{m:8s} {k:11s} " + " ".join(f"{rmse_kind.get((m, a, k), np.nan):7.2f}" for a in (0.0, 0.333, 0.667)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, k in zip(axes, ("convective", "stratiform")):
    for m in METHODS:
        a = [0.0, 0.333, 0.667]
        ax.plot([x * 15 for x in a], [rmse_kind.get((m, x, k), np.nan) for x in a], marker="o", label=m)
    ax.set_title(f"{k} times")
    ax.set_xlabel("minutes after degraded frame t0")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("RMSE on wet pixels [dBZ]")
axes[0].legend()
fig.tight_layout()
fig.savefig(OUT / "rmse_by_event_type.png", dpi=130)